# Closed-loop adaptive optics with a ZELDA wavefront sensor

This tutorial assembles a complete, deterministic adaptive-optics loop:

`source → pupil → static aberration → correcting DM → ZELDA WFS → detector → reconstructor → DM update`

The static aberration is generated by a second modal mirror so that its exact OPD is known. It plays the role of an atmosphere/NCPA realization projected onto the controllable Zernike space. The correction mirror is calibrated from detector images; the loop never uses the injected coefficients to compute its commands.

In [ ]:
import torch
import matplotlib.pyplot as plt

from fiatlux.core.grid import Grid
from fiatlux.core.spectrum import Band, Spectrum
from fiatlux.core.source import PlaneWave
from fiatlux.optics.detector import Detector
from fiatlux.optics.elements.deformable_mirror import (
    ActuatorGrid,
    DeformableMirror,
    ZernikeBasis,
)
from fiatlux.optics.elements.mask import CircularAperture, ZeldaMask
from fiatlux.optics.propagator import MFTPropagator
from fiatlux.system.interaction_matrix import InteractionMatrix
from fiatlux.system.optical_system import SerialSystem

## 1. Sampling and monochromatic source

The pupil and detector grids are deliberately small so the full calibration remains quick enough for an interactive tutorial and for CI.

In [ ]:
D = 1.0
wavelength = 1.65e-6
focal_length = 10.0
n_pixels = 64

pupil_grid = Grid(
    nx=n_pixels,
    ny=n_pixels,
    dx=D / n_pixels,
    dy=D / n_pixels,
)
focal_grid = Grid(
    nx=n_pixels,
    ny=n_pixels,
    dx=focal_length * wavelength / D / 4,
    dy=focal_length * wavelength / D / 4,
)

spectrum = Spectrum(
    magnitude=0,
    band=Band(
        central_wavelength=wavelength,
        delta_wavelength=0.0,
        f0=368.0,
    ),
    samples=1,
)
source = PlaneWave(spectrum=spectrum)

## 2. Pupil, aberration and correcting mirror

Both mirrors use the same eight-mode Zernike basis. The first one injects an unknown static wavefront; only the second one is passed to the interaction-matrix calibration.

In [ ]:
aperture = CircularAperture(grid=pupil_grid, radius=D / 2)
basis = ZernikeBasis(pixel_grid=pupil_grid, n=8)
actuator_grid = ActuatorGrid(
    n_actuators_x=1,
    n_actuators_y=1,
    pitch=D,
)

aberration = DeformableMirror(
    grid=pupil_grid,
    actuator_grid=actuator_grid,
    pixel_grid=pupil_grid,
    control_basis=basis,
    stroke=500e-9,
)
correcting_dm = DeformableMirror(
    grid=pupil_grid,
    actuator_grid=actuator_grid,
    pixel_grid=pupil_grid,
    control_basis=basis,
    stroke=500e-9,
)

## 3. ZELDA wavefront sensor and detector

ZELDA converts small pupil-plane phase errors into intensity variations. The detector is noiseless here so the example isolates the calibration and feedback logic.

In [ ]:
pupil_to_focal = MFTPropagator(
    focal_length=focal_length,
    output_grid=focal_grid,
)
zelda_mask = ZeldaMask(
    grid=focal_grid,
    radius=focal_length * wavelength / D,
    well_depth=wavelength / 4,
)
back_to_pupil = MFTPropagator(
    focal_length=focal_length,
    output_grid=pupil_grid,
)

detector = Detector(
    grid=pupil_grid,
    photon_noise=False,
    readout_noise_variance=0,
    dark_current=0,
)

ao_system = SerialSystem(
    elements=[
        aperture,
        aberration,
        correcting_dm,
        pupil_to_focal,
        zelda_mask,
        back_to_pupil,
    ]
)


def acquire_wfs_image():
    ao_system.run(source=source, detector=detector)
    return detector.image_buffer.clone()

## 4. Interaction and control matrices

Calibration is performed around a flat incoming wavefront. Push-pull finite differences give the detector response per metre of DM command. The filtered SVD pseudo-inverse maps an image residual back to modal OPD corrections.

In [ ]:
reference_image = acquire_wfs_image()

calibration = InteractionMatrix(
    dm=correcting_dm,
    acquiring_function=acquire_wfs_image,
    poke_amplitude=10e-9,
)
interaction_matrix = calibration.calibrate_push_pull(verbose=False)
control_matrix = calibration.compute_control_matrix(rcond=1e-4)

print("Interaction matrix:", tuple(interaction_matrix.shape))
print("Retained modes:", calibration.effective_rank)
print("Condition number:", f"{calibration.condition_number:.2f}")

## 5. Inject a static aberration

Only three non-piston Zernike coefficients are excited. Their values are shown for reproducibility, but the loop below observes only the detector image.

In [ ]:
injected_commands = torch.zeros_like(aberration.commands)
injected_commands[1] = 80e-9
injected_commands[3] = -50e-9
injected_commands[5] = 35e-9
aberration.commands = injected_commands

pupil_support = aperture.transmission.to(torch.bool)


def residual_opd():
    opd = (aberration.opd + correcting_dm.opd)[pupil_support]
    return opd - opd.mean()


initial_opd = residual_opd().detach().clone()
initial_image = acquire_wfs_image().detach().clone()
print("Initial residual RMS:", f"{initial_opd.square().mean().sqrt() * 1e9:.2f} nm")

## 6. Close the loop

At each iteration, the reference detector image is subtracted, the control matrix estimates the corrective increment, and an integrator with gain 0.6 updates the DM.

In [ ]:
gain = 0.6
n_iterations = 10
rms_history_nm = []

for iteration in range(n_iterations + 1):
    opd = residual_opd().detach()
    rms_history_nm.append(float(opd.square().mean().sqrt() * 1e9))

    if iteration == n_iterations:
        break

    image_residual = (acquire_wfs_image() - reference_image).flatten()
    command_increment = control_matrix @ image_residual
    correcting_dm.commands = correcting_dm.commands - gain * command_increment

final_opd = residual_opd().detach().clone()
final_image = acquire_wfs_image().detach().clone()

print("Final residual RMS:", f"{rms_history_nm[-1]:.3f} nm")
print("Reduction factor:", f"{rms_history_nm[0] / rms_history_nm[-1]:.0f}×")

assert rms_history_nm[-1] < 0.05 * rms_history_nm[0]

## 7. Convergence and before/after diagnostics

The RMS is computed from the physical residual OPD inside the pupil. It is used only as an evaluation metric, not as an input to the controller.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].semilogy(rms_history_nm, "o-")
axes[0].set_xlabel("Closed-loop iteration")
axes[0].set_ylabel("Residual OPD RMS [nm]")
axes[0].set_title("Loop convergence")
axes[0].grid(True)

limit_nm = float(initial_opd.abs().max() * 1e9)
for ax, opd, title in [
    (axes[1], initial_opd, "Initial residual OPD"),
    (axes[2], final_opd, "Final residual OPD"),
]:
    full_opd = torch.full(pupil_grid.shape, torch.nan)
    full_opd[pupil_support] = opd * 1e9
    image = ax.imshow(
        full_opd.cpu(),
        origin="lower",
        vmin=-limit_nm,
        vmax=limit_nm,
        cmap="RdBu_r",
    )
    ax.set_title(title)
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")

fig.colorbar(image, ax=axes[1:], label="OPD [nm]", shrink=0.85)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, image, title in [
    (axes[0], initial_image, "Initial ZELDA image"),
    (axes[1], final_image, "Final ZELDA image"),
]:
    displayed = ax.imshow(image.detach().cpu(), origin="lower")
    ax.set_title(title)
    fig.colorbar(displayed, ax=ax, label="Detector signal")

plt.show()

## What this example establishes

- The interaction matrix is measured from the optical system rather than supplied analytically.
- The control matrix is a filtered SVD pseudo-inverse.
- The controller uses only the ZELDA detector residual.
- The independently computed physical OPD RMS verifies convergence.
- Realistic turbulence, temporal delay and detector noise can be added without changing the loop structure.